In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import time
import pickle
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from scipy.stats import norm

from b_Closed_form import *
from c_MC import *
from d_FDM import *
from e_0_generate import *
from e_1_run_cvae import * 
from e_2_CVAE import *

# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)
# S0, K, r, kappa, theta, xi, rho, Y0, T = Hes_eta
Hes_eta = (S0, K, r, 2, 0.05, 0.5, -0.7, 0.05, T)

B = 0.8 # down-and-out must B < S0 and B < K
model_type = 'bs' # hes or bs
barr_type = 'van' # van or barr
opt_type = 'call' # call or put
chunk_dir = f"/mnt/d/bs_chunks_correction/" if model_type == 'bs' else f"/mnt/d/hes_chunks_correction/"


if not((B < S0) & (B < K)):
    raise ValueError("down-and-out : B should be smaller than S0 and K")

if not(opt_type == 'call' or  opt_type == 'put'):
    raise ValueError("option_type must be 'call' or 'put'")

if not(barr_type == 'van' or  barr_type == 'barr'):
    raise ValueError("barr_type must be 'van' or 'barr'")

if not(model_type == 'hes' or  model_type == 'bs'):
    raise ValueError("model_type must be 'hes' or 'bs'")

In [ ]:
from f_robust import (
    ROBUSTNESS_VARIABLES_MAIN,
    build_sampling_summary,
    plot_robustness,
    run_robustness,
    save_robustness_results,
)

robustness_variables_main = ROBUSTNESS_VARIABLES_MAIN


In [ ]:
robust_config = {
    "models": tuple(ROBUSTNESS_VARIABLES_MAIN.keys()),
    "barrier": B,
    "barr_types": ("van", "barr"),
    "option_types": ("call", "put"),
    "run_fdm": True,
    "run_mc": False,
    "n_samples_list": (1_000, 10_000, 100_000),
    "mc_repeats": 50,
    "mc_dt": 0.001,
}


In [ ]:
robust_results = run_robustness(
    BS_eta,
    Hes_eta,
    robustness_variables_main,
    **robust_config,
)

pickle_path, csv_path = save_robustness_results(robust_results)
print("raw results:", pickle_path)
print("summary    :", csv_path)

display(build_sampling_summary(robust_results))

for model_name in robust_config["models"]:
    for barr_type in robust_config["barr_types"]:
        plot_robustness(
            robust_results,
            model_name,
            barr_type,
            robustness_variables_main,
        )
